# Image Processing

> 📘 **Python Mastery** · Module 15 — Computer Vision · Lesson 2/5

Raw pixels are rarely ready: too dark, too noisy, full of background clutter. This lesson is the
classic "make images usable" toolbox — brightness and contrast, thresholding, blurring, sharpening
and morphology — the same preprocessing that still sits in front of most production vision systems.

## 🎯 Learning Objectives

- **Apply** point operations (brightness, contrast, gamma correction via a lookup table) safely
- **Read** an image histogram with `np.histogram` and predict what an operation will do to it
- **Binarize** documents with global, inverse and Otsu's automatic thresholding
- **Choose** the right smoothing filter — box vs Gaussian vs median — for each noise type
- **Sharpen** blurry images using convolution kernels (unsharp masking)
- **Clean** binary shapes with erosion, dilation, opening and closing, and extract outlines
- **Assemble** a complete pipeline that turns a noisy scanned page into clean black-and-white text

## 0. Our Running Example: A Scanned Page

We'll reuse one synthetic "document" throughout the lesson — white paper, dark handwriting-style
text — built programmatically so every result is reproducible. First, a helper to look at images,
and the document itself.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

Path("sample_data").mkdir(exist_ok=True)

def make_document(w=420, h=280):
    """White page with several lines of 'handwriting' (dark text)."""
    page = np.full((h, w), 235, dtype=np.uint8)
    lines = ["Invoice #0042 - Dhaka", "", "3 x notebooks ..... 240 tk",
             "1 x backpack ...... 950 tk", "Shipping .......... 60 tk",
             "TOTAL ............ 1250 tk"]
    y = 46
    for line in lines:
        cv2.putText(page, line, (28, y), cv2.FONT_HERSHEY_SIMPLEX,
                    0.55, (25, 25, 25), 2, cv2.LINE_AA)
        y += 34
    return page

doc = make_document()
cv2.imwrite("sample_data/page_clean.png", doc)

plt.figure(figsize=(5.5, 3.6))
plt.imshow(doc, cmap="gray", vmin=0, vmax=255)
plt.title("Our synthetic scanned page")
plt.axis("off")
plt.show()

## 1. Brightness & Contrast (Point Operations)

A **point operation** maps each pixel through the same function `new = f(old)`, ignoring neighbours.
Brightness shifts everything (`f(x) = x + b`); contrast scales distances from the middle
(`f(x) = a·x`, `a > 1` stretches, `a < 1` flattens). Remember lesson 1: use `cv2.add` / clipping so
values saturate instead of wrapping around.

**Syntax:**
```python
bright  = cv2.add(img, 40)                                   # shift up, saturating
contrast = np.clip(img.astype(np.float32) * 1.5, 0, 255).astype(np.uint8)
```

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

doc = cv2.imread("sample_data/page_clean.png", cv2.IMREAD_GRAYSCALE)

dim      = cv2.add(doc, -70)                                  # underexposed version
rescued  = cv2.add(dim, 70)                                   # brightness rescue
flat     = (0.5 * doc.astype(np.float32)).astype(np.uint8)     # low contrast
punchy   = np.clip((doc.astype(np.float32) - 128) * 1.8 + 128, 0, 255).astype(np.uint8)

fig, axes = plt.subplots(2, 4, figsize=(13, 4.6))
for j, (im, ttl) in enumerate([(doc, "original"), (dim, "-70 brightness"),
                               (flat, "contrast x0.5"), (punchy, "contrast x1.8")]):
    axes[0, j].imshow(im, cmap="gray", vmin=0, vmax=255); axes[0, j].set_title(ttl, fontsize=9)
    hist, _ = np.histogram(im, bins=256, range=(0, 256))
    axes[1, j].plot(hist, color="black", lw=0.9)
    axes[1, j].set_xlim(0, 255); axes[1, j].set_yticks([])
for ax in axes[0]:
    ax.axis("off")
for ax in axes[1]:
    ax.set_xlabel("pixel value", fontsize=8)
plt.suptitle("Top: image   Bottom: its histogram (np.histogram)")
plt.tight_layout()
plt.show()

print("Contrast stretching moves histogram bars APART; brightness slides them LEFT/RIGHT.")

## 2. Gamma Correction with a Lookup Table

Human eyes are nonlinear: we notice shadow detail far more than highlight detail. **Gamma
correction** reshapes midtones with a power law, $new = 255\,(old/255)^{\gamma}$:
$\gamma < 1$ brightens shadows (rescuing a dark photo), $\gamma > 1$ darkens them. Because there are
only 256 possible inputs, we precompute all answers once into a **lookup table** and apply them with
`cv2.LUT` — one table lookup per pixel instead of a power computation.

**Syntax:**
```python
gamma = 0.5
lut = (np.arange(256) / 255.0) ** gamma * 255        # 256-entry answer key
lut = np.clip(lut, 0, 255).astype(np.uint8)
out = cv2.LUT(img, lut)
```

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

doc_dark = cv2.add(cv2.imread("sample_data/page_clean.png", cv2.IMREAD_GRAYSCALE), -90)

def gamma_lut(g):
    lut = (np.arange(256, dtype=np.float32) / 255.0) ** g * 255.0
    return np.clip(lut, 0, 255).astype(np.uint8)

fixed = cv2.LUT(doc_dark, gamma_lut(0.45))

fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
axes[0].imshow(doc_dark, cmap="gray", vmin=0, vmax=255); axes[0].set_title("Dark scan (brightness -90)")
axes[1].imshow(fixed, cmap="gray", vmin=0, vmax=255);    axes[1].set_title("After gamma = 0.45 (LUT)")
xs = np.arange(256)
axes[2].plot(xs, xs, "--", color="gray", label="identity")
axes[2].plot(xs, gamma_lut(0.45), color="crimson", label="gamma 0.45")
axes[2].plot(xs, gamma_lut(2.0), color="navy", label="gamma 2.0")
axes[2].set_xlabel("input"); axes[2].set_ylabel("output"); axes[2].legend(fontsize=8)
axes[2].set_title("The lookup-table curves")
for ax in axes[:2]:
    ax.axis("off")
plt.tight_layout()
plt.show()

print("Same 256-entry table works on ANY grayscale or BGR image - cheap per-pixel math.")

## 3. Thresholding: Turning Gray into Black & White

Thresholding keeps only two levels — perfect for text, barcodes and silhouettes.
`cv2.threshold` returns `(computed_threshold, binary_image)`; you pick the style with flags:

| Flag | Pixels above thresh become |
|------|----------------------------|
| `THRESH_BINARY` | 255 (rest 0) |
| `THRESH_BINARY_INV` | 0 (rest 255) |
| `THRESH_TRUNC` | capped at thresh |
| `THRESH_TOZERO` | kept, below set to 0 |

Passing `cv2.THRESH_OTSU` (OR-ed with a flag) makes OpenCV **find the threshold for you** by
minimizing within-class variance of the histogram — ideal when you don't know the lighting level.
Otsu needs a roughly **two-peaked (bimodal)** histogram, which is exactly what a document has.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

doc = cv2.imread("sample_data/page_clean.png", cv2.IMREAD_GRAYSCALE)

_, hard = cv2.threshold(doc, 127, 255, cv2.THRESH_BINARY)          # manual cut at 127
_, inv  = cv2.threshold(doc, 127, 255, cv2.THRESH_BINARY_INV)      # inverted: ink becomes white
otsu_val, otsu = cv2.threshold(doc, 0, 255,
                               cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)  # auto threshold

fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.6))
for ax, im, ttl in zip(axes, [hard, inv, otsu],
                       ["THRESH_BINARY @ 127", "THRESH_BINARY_INV @ 127",
                        f"OTSU auto (thresh = {otsu_val:.0f})"]):
    ax.imshow(im, cmap="gray"); ax.set_title(ttl, fontsize=9); ax.axis("off")
plt.suptitle("Same page, three threshold styles")
plt.show()

print("Otsu picked threshold", int(otsu_val),
      "- close to our hand-picked 127, but found automatically.")
print("Inverted output is what OCR libraries expect: ink = white (255).")

## 4. Smoothing: Box vs Gaussian vs Median

Noise is random per-pixel junk; smoothing replaces each pixel with information from its
neighbourhood. The three workhorse filters behave very differently:

| Filter | How it works | Best when |
|--------|--------------|-----------|
| `cv2.blur` (box) | plain average of the k×k window | quick softening; edges get mushy |
| `cv2.GaussianBlur` | weighted average, centre counts most | general pre-processing before detectors |
| `cv2.medianBlur` | take the *middle* value of the window | **salt-and-pepper** impulses; preserves edges |

Kernel sizes must be **odd** (so there's a true centre pixel): 3, 5, 7… Larger = smoother = blurrier.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)                       # seeded -> reproducible noise
doc = cv2.imread("sample_data/page_clean.png", cv2.IMREAD_GRAYSCALE).copy()

sp = rng.random(doc.shape)                            # salt-and-pepper mask
doc[sp < 0.03] = 0                                    # pepper
doc[sp > 0.97] = 255                                  # salt
cv2.imwrite("sample_data/page_noisy.png", doc)

box     = cv2.blur(doc, (5, 5))                       # box average
gauss   = cv2.GaussianBlur(doc, (5, 5), 0)            # sigma=0 -> derived from ksize
median  = cv2.medianBlur(doc, 5)                      # winner for impulse noise

fig, axes = plt.subplots(2, 2, figsize=(10, 7))
for ax, im, ttl in zip(axes.ravel(), [doc, box, gauss, median],
                       ["Noisy input (salt & pepper)", "Box blur 5x5",
                        "Gaussian blur 5x5", "Median blur 5x5"]):
    ax.imshow(im, cmap="gray", vmin=0, vmax=255)
    ax.set_title(ttl, fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()

print("Look closely: box & Gaussian SMEAR each dot into a gray blob;")
print("median DELETES the dots outright because 0/255 are extreme outliers of the window.")

> 🔍 **Under the Hood:** `filter2D`-style convolutions in OpenCV are implemented as
> **separable filters** when possible: a 2-D Gaussian kernel factors into two 1-D passes (rows, then
> columns), turning O(k²) work per pixel into O(2k). The median filter can't be factored that way —
> it needs actual sorting per window — which is why it's slower but unbeatable against impulse noise.
> Also note: OpenCV's `filter2D` computes **correlation**, not flipped-kernel convolution; symmetric
> kernels like Gaussians make the difference invisible, but asymmetric kernels won't be mirrored.

## 5. Sharpening: Unsharp Masking

Blurring removes detail, so adding the *difference* between sharp and blurred back onto the image
restores crisp edges — that's the classic **unsharp mask**. In kernel form it collapses to one
weighted convolution: centre weight `5`, immediate neighbours `-1`. Stronger sharpening = bigger
centre weight (try 9 instead of 5).

**Syntax:**
```python
kernel = np.array([[ 0, -1,  0],
                   [-1,  5, -1],
                   [ 0, -1,  0]], dtype=np.float32)
sharp = cv2.filter2D(img, ddepth=-1, kernel=kernel)
```

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

doc = cv2.imread("sample_data/page_clean.png", cv2.IMREAD_GRAYSCALE)
blurry = cv2.GaussianBlur(doc, (7, 7), 2.5)           # pretend the scanner was out of focus

sharpen_kernel = np.array([[0., -1., 0.],
                           [-1., 5., -1.],
                           [0., -1., 0.]])
sharp = cv2.filter2D(blurry, -1, sharpen_kernel)       # -1: keep source depth

# Equivalent explicit form: sharp = original + amount * (original - blurred)
amount = 1.0
unsharp = cv2.addWeighted(blurry, 1 + amount, cv2.GaussianBlur(blurry, (5, 5), 1.5), -amount, 0)

fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.6))
for ax, im, ttl in zip(axes, [blurry, sharp, unsharp],
                       ["Blurry input", "Kernel sharpen (center=5)", "Unsharp mask, amount=1.0"]):
    ax.imshow(im, cmap="gray", vmin=0, vmax=255)
    ax.set_title(ttl, fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

print("Push the center weight to 9 for stronger effect - or watch halos appear around letters.")

## 6. Morphology: Surgery for Binary Images

On a black-and-white image, erosion shrinks white regions and dilation grows them, using a small
structuring element (**kernel**) as the probe. Alone they're destructive; **combined** they become
precise tools:

| Operation | Recipe | Effect |
|-----------|--------|--------|
| Erosion `erode` | min over window | eats away thin white noise |
| Dilation `dilate` | max over window | grows regions, fills pinholes |
| **Opening** `MORPH_OPEN` | erode → dilate | removes small white specks, keeps size |
| **Closing** `MORPH_CLOSE` | dilate → erode | seals small holes, keeps size |
| Gradient `MORPH_GRADIENT` | dilate − erode | the OUTLINE of every region |

All four composites come from one call: `cv2.morphologyEx(img, op, kernel)`.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

rng = np.random.default_rng(7)
binary = np.zeros((180, 320), dtype=np.uint8)          # two solid shapes...
cv2.rectangle(binary, (30, 40), (120, 140), 255, -1)
cv2.circle(binary, (230, 90), 55, 255, -1)
noise = rng.random(binary.shape)                       # ...plus specks and pinholes
binary[noise < 0.02] = 255
binary[(noise > 0.75) & (binary == 255)] = 0

kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
steps = [("Noisy binary", binary),
         ("erode", cv2.erode(binary, kernel)),
         ("dilate", cv2.dilate(binary, kernel)),
         ("open (erode->dilate)", cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)),
         ("close (dilate->erode)", cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel))]

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for ax, (ttl, im) in zip(axes, steps):
    ax.imshow(im, cmap="gray"); ax.set_title(ttl, fontsize=9); ax.axis("off")
plt.suptitle("Elliptical kernel 7x7 - one probe, five outcomes")
plt.tight_layout()
plt.show()

outline = cv2.morphologyEx(cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel),
                           cv2.MORPH_GRADIENT, kernel)  # clean first, THEN trace outlines
plt.figure(figsize=(5, 3))
plt.imshow(outline, cmap="gray")
plt.title("MORPH_GRADIENT after opening: instant outlines")
plt.axis("off")
plt.show()

## 7. Gradients Preview: Sobel & Laplacian

Where brightness changes sharply, there is usually an **edge**. The Sobel operators measure change
along one axis each (`dx=1,dy=0` → vertical edges; `dx=0,dy=1` → horizontal); combining their
magnitudes gives an edge strength map. The Laplacian takes the second derivative — one operator,
all directions, but noisier. These are the ingredients Canny will orchestrate next lesson.

**Syntax:**
```python
gx = cv2.Sobel(img, cv2.CV_64F, 1, 0, ksize=3)     # derivative along x
gy = cv2.Sobel(img, cv2.CV_64F, 0, 1, ksize=3)
mag = cv2.magnitude(gx, gy)                          # sqrt(gx^2 + gy^2)
lap = cv2.Laplacian(img, cv2.CV_64F)
```

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

shapes = np.zeros((200, 300), dtype=np.uint8)
cv2.rectangle(shapes, (35, 45), (135, 150), 255, -1)
cv2.circle(shapes, (215, 95), 52, 255, -1)
smoothed = cv2.GaussianBlur(shapes, (5, 5), 0)       # smooth first: derivatives hate noise

gx = cv2.Sobel(smoothed, cv2.CV_64F, 1, 0, ksize=3)
gy = cv2.Sobel(smoothed, cv2.CV_64F, 0, 1, ksize=3)
mag = np.sqrt(gx**2 + gy**2)
lap = cv2.Laplacian(smoothed, cv2.CV_64F)

panels = [(smoothed, "Input"), (np.abs(gx), "|Sobel x|: vertical edges"),
          (np.abs(gy), "|Sobel y|: horizontal edges"),
          (mag, "Gradient magnitude"), (np.abs(lap), "|Laplacian|")]
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for ax, (im, ttl) in zip(axes, panels):
    ax.imshow(im, cmap="gray"); ax.set_title(ttl, fontsize=8.5); ax.axis("off")
plt.tight_layout()
plt.show()

print("CV_64F output keeps negative slopes; take np.abs to visualize.")

## 8. Putting It Together: Cleaning a Noisy Scan

Real preprocessing is a *pipeline*, and order matters. Our recipe for a messy scan:

1. **Median blur** — delete salt-and-pepper without smearing letter strokes
2. **Otsu threshold (inverted)** — snap to pure black & white, ink = white
3. **Closing** — rejoin letters broken by noise
4. **Opening** — erase surviving specks in the background

Each step assumes something about the previous one — that's why pipelines beat single magic calls.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

rng = np.random.default_rng(123)
clean = cv2.imread("sample_data/page_clean.png", cv2.IMREAD_GRAYSCALE)

# 0) simulate a bad scan: uneven brightness + gaussian hiss + dust
gradient = np.tile(np.linspace(210, 175, clean.shape[1]).astype(np.int16), (clean.shape[0], 1))
bad = np.clip(clean.astype(np.int16) - 40 + gradient, 0, 255).astype(np.uint8)
hiss = rng.normal(0, 14, bad.shape).astype(np.int16)          # gaussian sensor hiss
bad = np.clip(bad.astype(np.int16) + hiss, 0, 255).astype(np.uint8)
dust = rng.random(bad.shape)
bad[dust < 0.015] = 0
bad[dust > 0.985] = 255
cv2.imwrite("sample_data/page_bad_scan.png", bad)

# 1)-4) the pipeline
step1 = cv2.medianBlur(bad, 5)
_, step2 = cv2.threshold(step1, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
k3 = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
step3 = cv2.morphologyEx(step2, cv2.MORPH_CLOSE, k3)
step4 = cv2.morphologyEx(step3, cv2.MORPH_OPEN, k3)

ink_before = (step2 > 0).mean(); ink_after = (step4 > 0).mean()
print(f"Ink coverage: raw binarized {100*ink_before:.1f}% -> cleaned {100*ink_after:.1f}%")

titles = ["Bad scan", "1. median blur", "2. Otsu (inv)", "3. close", "4. open = final"]
fig, axes = plt.subplots(1, 5, figsize=(16, 3.2))
for ax, im, ttl in zip(axes, [bad, step1, step2, step3, step4], titles):
    ax.imshow(im, cmap="gray"); ax.set_title(ttl, fontsize=9); ax.axis("off")
plt.tight_layout()
plt.show()

## ⚠️ Common Mistakes & Gotchas

| Mistake | Problem | Fix |
|---------|---------|-----|
| `cv2.GaussianBlur(img, (6, 6), 0)` | Even kernel size → error ("ksize.width must be odd") | Use odd sizes: 3, 5, 7… |
| Box/Gaussian blur to remove salt-and-pepper | Each dot spreads into a gray blob | `cv2.medianBlur` deletes impulses |
| Thresholding before denoising | Noise survives as thousands of fake foreground pixels | Smooth first, threshold second |
| Otsu on unevenly lit photos | Single global cut fails on gradients/shadows | Consider `cv2.adaptiveThreshold` |
| Morphology directly on a grayscale photo | Erode/dilate act on intensity extremes, results look bizarre | Convert/darken to binary first |
| Forgetting `.astype(np.uint8)` after float math | `cv2.threshold` throws a dtype error | Widen → clip → cast back, always |

## 💡 Best Practices & Pro Tips

- **Inspect histograms, not just pictures.** A glance at the histogram tells you whether Otsu will
  work (two peaks) and whether your contrast stretch did anything.
- **Prefer median for impulse noise, Gaussian before edge detection.** They solve different problems;
  keeping both in your toolkit is cheaper than debugging mysterious artifacts.
- **Keep pipelines small and printable**: show an intermediate panel per step (as above) so you can
  see exactly where quality dies.
- **Tune kernels relative to feature size**: a stroke 2 px wide is destroyed by erosion with a 5 px
  kernel. Measure first, then choose the structuring element.
- 🤖 **AI-engineering relevance:** these exact operations are standard dataset preprocessing — deskew
  and denoise scans before OCR, normalize exposure before face recognition, morphologically clean
  segmentation masks before computing metrics. Deep models still eat cleaner inputs better.

## 📌 Summary

| Method | What it does | Example |
|--------|--------------|---------|
| `cv2.add(img, b)` | Brightness shift, saturating | `cv2.add(img, 40)` |
| `clip(a*x + b)` pattern | Contrast scaling, safe dtype | `np.clip(img*f, 0, 255).astype(np.uint8)` |
| `cv2.LUT(img, lut)` | Apply 256-entry curve (gamma) | `cv2.LUT(img, gamma_lut(0.5))` |
| `cv2.threshold(img, t, 255, flags)` | Binarize; `+THRESH_OTSU` auto-tunes t | `THRESH_BINARY_INV + THRESH_OTSU` |
| `cv2.medianBlur(img, k)` | Impulse-noise killer (odd k) | `cv2.medianBlur(noisy, 5)` |
| `cv2.GaussianBlur(img, (k,k), s)` | Weighted smooth before detectors | `cv2.GaussianBlur(img, (5,5), 0)` |
| `cv2.filter2D(img, -1, K)` | Custom convolution (sharpen…) | center-weight 5 kernel |
| `cv2.morphologyEx(img, op, K)` | OPEN/CLOSE/GRADIENT composites | `cv2.morphologyEx(bw, MORPH_CLOSE, k)` |
| `cv2.Sobel / Laplacian` | Derivatives → edge strength | `cv2.Sobel(img, cv2.CV_64F, 1, 0)` |

- Point operations reshape the histogram; thresholding collapses it to two spikes.
- Median ≠ average: only the median deletes salt-and-pepper outright.
- Opening removes white specks, closing fills black holes — both preserve object size.
- Pipeline order matters: denoise → threshold → morphological cleanup.

## 🔗 Next Lesson

Continue to **[03_Edges_Contours_Features](../03_Edges_Contours_Features/notes.ipynb)** — Canny
edges, contours, corner detection and template matching: finding *structures*, not just cleaning
pixels.